# Introduction to Control Engineering

**Learning Goals**
- Understand what a system is
    - understand system inputs and outputs
- Understand what a system model is
- Understand what a control system is
- Understand why control is needed
- Understand the differences between open-loop and closed-loop control
- See why open-loop control fails
- See how closed-loop (proportional) control can be beneficial

## Relevant lecture video

In [1]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://echo360.org/media/3270cf22-254b-4d1e-a3cb-6927d45fe5bc/public?autoplay=false&automute=false&currentMediaId=89d79b10-7db7-4674-b90f-946735c7140e" frameborder="0" allowfullscreen></iframe>')

/home/matvei/JupyterBasedControlEngineeringTextbook/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## What is a system?

For the purposes of this course, we will define systems as arrangements of physical components related in such a form as to act as an entire unit.

## What is a control system?

Control systems are arrangements of physical components related in such a form as to regulate another system.

**For example**, a person driving an automobile is a control system. Driver assistance systems are also control systems. 

## What is input and output?

The input is the stimulus, excitation, or command applied to a control system to produce a specified response from the control system.

The output is the response obtained from a control system.

## Open-loop and closed-loop control

Control systems can be either **open-loop** or **closed-loop**.

**Open-loop** control systems provide control actions independent of system output.
- For example, toasters typically use timers for control. Temperature or other physical characteristics of the environment inside the toaster are not measured, as time enough is sufficient to make a "good toast".

**Closed-loop** control systems provide control actions that depend on system output. Typically, those actions are a mathematical function of system output.
- For example, lane follow assist can be implemented as a function of the distance from the center of the lane as estimate by the vehicle's sensors (system output) to the center of the vehicle.

## System model

Physical systems need to be modeled in order to be analyzed. A **model** is an approximation of system behavior. A model shows the relationship between system input and output.

## Example Problem: A Block on Ice

Imagine a block sitting on a slippery (low friction) icy surface. We want to move the block to a desired **setpoint** position like $x = 10$ m and have it stay there.

We can apply a horizontal force $F$ to the block. The block obeys Newton's second law: $F = m a$. On ice, once the block starts moving, it will keep moving for a very long period of time unless we apply a force in the opposite direction.

Our goal is to design a **controller** that decides how much force to apply at each moment so that the block reaches the setpoint and stays at the setpoint.

### System model

The block and ice surface form a **second-order system**. The input to the system is the applied force $F(t)$. The output is the block position $x(t)$.

Newton’s second law $F = m a$ gives the equation of motion:

$$m \ddot x = F - b \dot x$$

where $\dot x$ is velocity, $\ddot x$ is acceleration, $m$ is mass, and $b$ is the friction coefficient.

The **state** of the system is the pair $(x, \dot x)$. Writing as two first-order equations:

$$
\begin{aligned}
\dot x &= v \\[4pt]
\dot v &= \frac{1}{m}\bigl(F - b\,v\bigr)
\end{aligned}
$$

In our simulation we use $m = 10$ kg and $b = 0.5$ N·s/m (light damping from ice friction).

### Import the libraries

In [2]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from scipy.integrate import solve_ivp
from IPython.display import display, clear_output
import ipywidgets as widgets

print("Libraries loaded.")

Libraries loaded.


### Simulation and animation helpers

In [11]:
def simulate_block(control_func, t_span=30.0, dt=1/240, friction=0.5, mass=10.0):
    def ode(t, state):
        x, v = state
        force = control_func(t, x, v)
        friction_force = -friction * v
        dxdt = v
        dvdt = (force + friction_force) / mass
        return [dxdt, dvdt]

    t_eval = np.arange(0, t_span, dt)
    sol = solve_ivp(ode, [0, t_span], [0.0, 0.0], t_eval=t_eval, method='RK45')
    forces = np.array([control_func(t, sol.y[0, i], sol.y[1, i]) for i, t in enumerate(sol.t)])
    return sol.t, sol.y[0], forces

def make_animation(times, positions, forces, title, arrow_scale=0.08, color='b'):
    n_frames = 30
    step = max(1, len(times) // n_frames)
    t = times[::step][:n_frames]
    y = positions[::step][:n_frames]
    f = forces[::step][:n_frames]
    x_max = max(25, np.ceil(max(positions) + 2))
    f_max = max(abs(forces)) if len(forces) else 1

    fig = plt.figure(figsize=(8, 5.5))
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.set_xlim(-1, x_max)
    ax1.set_ylim(-0.5, 3.5)
    ax1.set_aspect('equal')
    ax1.axhline(0, color='gray', linewidth=2)
    ax1.axvline(10, color='k', ls='--', alpha=0.4, linewidth=1.5, label='Setpoint = 10 m')
    ax1.set_ylabel('Block')
    ax1.legend(loc='upper right', fontsize=9)
    ax1.set_title(title, fontsize=11)
    block = Rectangle((0, 0), 2, 1, fc=color, ec='k', lw=2)
    ax1.add_patch(block)
    time_text = ax1.text(0.02, 0.88, '', transform=ax1.transAxes, fontsize=9)
    force_text = ax1.text(0.02, 0.78, '', transform=ax1.transAxes, fontsize=9, color=color)

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.set_xlim(0, 30)
    ax2.set_ylim(-1, x_max)
    ax2.axhline(10, color='k', ls='--', alpha=0.4)
    ax2.set(xlabel='Time (s)', ylabel='Position (m)')
    ax2.grid(alpha=0.3)
    line, = ax2.plot([], [], color=color, linewidth=2)

    fig.tight_layout()

    def init():
        block.set_xy((-1, 0))
        line.set_data([], [])
        time_text.set_text('')
        force_text.set_text('')
        return block, line, time_text, force_text

    def animate(i):
        xi = y[i]
        fi = f[i]
        block.set_xy((xi - 1, 0))
        line.set_data(t[:i], y[:i])
        time_text.set_text(f't = {t[i]:.1f} s')
        force_text.set_text(f'F = {fi:+.1f} N')
        return block, line, time_text, force_text

    ani = FuncAnimation(fig, animate, frames=len(t),
                        init_func=init, blit=True, interval=100)
    plt.close(fig)
    return HTML(ani.to_jshtml())

print("Helpers defined.")

Helpers defined.


---

### Open-Loop Control

In **open-loop control**, the controller decides the force based only on the desired setpoint. It does **not** use any information about the block's actual position.

A simple open-loop strategy: apply a constant force $F = K \cdot r$, where $r = 10$ m is the setpoint and $K$ is a **gain** we can adjust.

**Click Run to see it in action!**

In [13]:
K_slider = widgets.FloatSlider(min=0.0, max=3.0, step=0.05, value=1.0,
                                description='Gain K:', style={'description_width': 'initial'})
run_btn = widgets.Button(description='Run', button_style='primary')
ol_out = widgets.Output()

def on_run_ol(b):
    K = round(K_slider.value, 2)
    ctrl = lambda t, x, v: K * 10.0
    times, positions, forces = simulate_block(ctrl)
    with ol_out:
        clear_output(wait=True)
        display(make_animation(times, positions, forces,
                               f'Open-Loop K={K:.2f}', color='b'))

run_btn.on_click(on_run_ol)
display(widgets.VBox([K_slider, run_btn, ol_out]))

**Observation.** With open-loop control the block accelerates until the ice friction matches the applied force: it reaches a constant terminal velocity. The block never settles at the setpoint because the controller doesn't know where the block actually is.

---

### Closed-Loop (Proportional) Control

In **closed-loop control**, the controller uses the current position to decide the force. The simplest closed-loop strategy is **proportional (P) control**:

$$F(t) = K_p \, \bigl(r - x(t)\bigr)$$

Here $e(t) = r - x(t)$ is the **error**, the difference between where we want the block and where it actually is.

Ice friction is included in the physics (proportional to velocity), so the block will experience some **damping**. With P-control and damping, the block can settle at the setpoint, but it will **oscillate** on the way there if the damping is light.

**Click Run!**

In [14]:
Kp_slider = widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0,
                                 description='Kp:', style={'description_width': 'initial'})
run_btn2 = widgets.Button(description='Run', button_style='primary')
pc_out = widgets.Output()

def on_run_pc(b):
    Kp = round(Kp_slider.value, 1)
    ctrl = lambda t, x, v: Kp * (10.0 - x)
    times, positions, forces = simulate_block(ctrl)
    with pc_out:
        clear_output(wait=True)
        display(make_animation(times, positions, forces,
                               f'P-Control Kp={Kp:.1f}', color='r'))

run_btn2.on_click(on_run_pc)
display(widgets.VBox([Kp_slider, run_btn2, pc_out]))

**Observation.** The proportional controller pulls the block toward the setpoint.

With ice friction providing light damping, the block exhibits **underdamped** response: it overshoots, swings back, overshoots less, and eventually settles at the setpoint. Lower $K_p$ gives a sluggish response with little overshoot; higher $K_p$ makes it faster but more oscillatory.